In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Machine Learnin Project # 
The goal is to predict the temperature in Bern in 12h, 24h and 48h using a dataset containing different measures from 10 different swiss cities. This is a regression task and will be approached using different models. 
The error measure used to evaluate our prediction is the mean absolute error (MAE).
Our training dataset is the "train (2)" file and our testing dataset is the "test" file, both in the Kaggle input section.

# 1) Introduction 

## 1.1) Importing packages

First, we import everything we will need in this project.

In [2]:
# Encoder, scaler, search...
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

# Models 
from sklearn.linear_model import LinearRegression, Ridge , Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor

# Maths
from sklearn.metrics import mean_absolute_error
from math import sqrt
import itertools

# Plotting
import seaborn as sns
import matplotlib.pyplot as plt

# Ignore warnings 
import warnings
warnings.filterwarnings("ignore")

## 1.2) Preparation : data cleaning functions

### 1.2.1) Names and limits

We need to prepare some data structures that we will use in the data cleaning part.

The "COLNAMES" dictionary will help us change the column names. The "TARGET_CADIDATES" list is used to identify the three targets. Also, set a "physical_limits" dictionary to check for impossible values. 

In [3]:
# All measures are to be considered during a given hour.
COLNAMES = {
    "fkl010h0": "avg_wind",              # average wind speed (km/h).
    "fkl010h3": "max_wind",              # maximum wind speed (km/h).
    "gre000h0": "avg_sol_rad",           # average solar radiation (W/m²).
    "pp0qffh0": "avg_air_press_sea",     # average air pressure at sea level (hPa).
    "prestah0": "avg_air_press_bar",     # average air pressure at station level (hPa).
    "rre150h0": "sum_rain",              # total precipitation (mm).
    "sre000h0": "sum_sun_time",          # total sunshine duration (minutes).
    "tre200h0": "avg_air_temp",          # average air temperature (celsius). 
    "ure200h0": "avg_air_hum",           # average air humidity (%).
}
TARGET_CANDIDATES = [
    "target_tre200h0_plus12h",
    "target_tre200h0_plus24h",
    "target_tre200h0_plus48h",
]
physical_limits = {
    "avg_wind": (0, 100),                # Wind speed cannot be negative.
    "max_wind": (0, 120),                # Fastest wind ever is 113.
    "avg_sol_rad": (0, 1400),            # Solar radiation cannot be negative. 
    "avg_air_press_sea" : (760, 1100),   # Limits put considering the altitude of all cities. 
    "avg_air_press_bar" : (760, 1100),   # Limits put considering the altitude of all cities. 
    "sum_rain": (0, 500),                # Rain cannot be negative.
    "sum_sun_time": (0, 60),             # Cannot be more than 60min/h.
    "avg_air_temp": (-50, 45),           # Temperature extreme range (Celsius).
    "avg_air_hum": (0, 100)              # humidity % cannot exceed 100 .
}


### 1.2.2) Functions

We define 6 data cleaning functions : 
- rename_columns() : uses the dictionary (COLNAMES) to rename columns into more understandable names.
- check_impossible_values() : checks for values outside of the limits in our dictionary (physical_limits).
- create_avg_temp_season_hour() : creates a column containing the average temperature per hour per season (in our case, for Bern, but it is a generalized function).
- encode_hour_and_season() : encodes hour as cyclical value and season as 4 dummy variables.
- scaling_data() : uses a StandardScaler to scale values in the dataset. Every column will have a mean of 0 and a standard deviation of 1. 
- high_corr_filter() : drops highy correlated columns (=more than the treshold). We put a treshold at 99% which means that it drops 1 of 2 columns that are correlated at more than 99%. We do this because the two columns are basically identical and won't help the accuracy of our models. 

In [4]:
def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renames columns with the predefined names dictionary. 
    """

    out = df.copy()

    for col in list(out.columns):
        
        # Skip columns hour and season.
        if col in ["hour", "season"]:
            continue
            
        # If the column has a prefix and station code.
        if "_" in col:
            left, right = col.split("_", 1)
            
            # If the prefix is in our dictionary, rename it ("left") and put back the city ("right")
            if left in COLNAMES:
                out.rename(columns={col: f"{COLNAMES[left]}_{right}"}, inplace=True)
                
    # Rename columns without a city name (Bern)
    out.rename(columns={
                   'tre200h0': 'target_temp',       
                   'target_tre200h0_plus12h': 'target_temp_plus12',
                   'target_tre200h0_plus24h': 'target_temp_plus24',
                   'target_tre200h0_plus48h': 'target_temp_plus48'}, inplace=True)
    return out

In [5]:
def check_impossible_values(df : pd.DataFrame) -> pd.DataFrame:
    """
    Checking if our data contains impossible values (ie. values outside of the limits)
    """
    # Get the min and max for each column 
    check_df = df.describe().loc[['max','min'],:] 

    # Check every column but not the targets (last 3 columns).
    for col in check_df.columns[:-3]:                 

        # Skip the 'Id' column
        if col=='Id':                                 
            continue

        #  for "target_temp" and "avg_air_temp_lag24h" : set the limits for temperatures.
        elif (col.startswith('target')) or (col=='avg_air_temp_lag24h'):  
            min_ = physical_limits['avg_air_temp'][0]           
            max_ = physical_limits['avg_air_temp'][1]

        # For the rest of the columns : 
        else:   
            
            # Get limits
            min_ = physical_limits[col[:-4]][0]       
            max_ = physical_limits[col[:-4]][1]       

        # Check that values are in between limits. 
        value_max = check_df.loc['max', col] 
        value_min = check_df.loc['min', col]
        if (value_max>max_) :
            print(f"Max problem on column {col}")
            print(f"Max Limit: {max_}, value{value_max} ")
        if (value_min<min_) : 
            print(f"Min problem on column {col}")
            print(f"Min Limit: {min_}, value{value_min} ")
    
    return df

In [6]:
def create_avg_temp_season_hour(df : pd.DataFrame,
                                city_temp_cols : list,
                                name : str, 
                                show_plot : bool = False ) -> pd.DataFrame:
    """
    Creates a new column that displays the mean temeprature PER hour and PER season. 
    Show_plot creates a plot to confirm that it has created a coherent column. 
    """
    # Make a copy
    out = df.copy()

    # Row-wise mean temperature across all n cities 
    out["mean_temp"] = out[city_temp_cols].mean(axis=1)

    # Average mean temp by season AND hour. Then, rename the column.  
    avg_table = (
        out.groupby(["season", "hour"])["mean_temp"]
          .mean()
          .reset_index()
          .rename(columns={"mean_temp": f"avg_temp_season_hour_{name}"})
    )
    out = out.drop('mean_temp',axis=1)

    # Merge back
    out = out.merge(avg_table, on=["season", "hour"], how="left")

    # Plot only if show_plot=True.
    if show_plot: 
        
        # Create pivot table: rows = season, columns = hour
        pivot = out.pivot_table(
            index="season",
            columns="hour",
            values=f"avg_temp_season_hour_{name}"
        )
        
        # Plot heatmap 
        plt.figure(figsize=(14, 6))
        plt.imshow(pivot, aspect='auto')
        plt.colorbar(label='Avg Temp (°C)')
        
        # Tick labels
        plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=90)
        plt.yticks(range(len(pivot.index)), pivot.index)

        # Details
        plt.title("Average Temperature by Season and Hour (check)")
        plt.xlabel("Hour")
        plt.ylabel("Season")
        plt.tight_layout()
        plt.show()
        
        pivot
    return out

In [7]:
def encode_hour_and_season(df: pd.DataFrame) -> pd.DataFrame:
    """
    Encodes "hour" using sin and cos for cyclical properties.
    Encodes "season" as 4 dummy variables.
    """
    out = df.copy()

    # Hour : cyclic encoding, create two new numeric features from hour
    h = out["hour"]                   
    out["hour_sin"] = np.sin(2 * np.pi * h / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * h / 24.0)
    out = out.drop('hour', axis=1)

    # Season : one-hot encoding
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoder.fit(out[['season']])
    encoded_array = encoder.transform(out[['season']])
    encoded_cols = list(encoder.get_feature_names_out(['season']))
    
    out[encoded_cols] = encoded_array
    out = out.drop('season', axis=1)

    return out


In [8]:
def scaling_data(df_train : pd.DataFrame,
                 df_test : pd.DataFrame) -> pd.DataFrame:
    """
    Scales data using StandardScaler(). 
    """
    scaled_df_train = df_train.copy()
    scaled_df_test = df_test.copy()
    
    # Columns that should not be touched.
    NoScale_cols_train = ['hour_sin','hour_cos','season_Autumn','season_Spring','season_Summer','season_Winter', 'target_temp_plus12', 'target_temp_plus24', 'target_temp_plus48']
    NoScale_cols_test =  ['hour_sin','hour_cos','season_Autumn','season_Spring','season_Summer','season_Winter']
    
    # Columns that should be standardized : the rest. 
    StandardScale_cols = [col for col in df_test.columns if (col not in NoScale_cols_train)]

    # Create a DataFrame. 
    scaled_df = pd.DataFrame(index = df_train.index) 

    # Scaling : fit to training, transform both. 
    std_scaler = StandardScaler()
    std_scaler.fit(df_train[StandardScale_cols])
    scaled_df_train[StandardScale_cols] = std_scaler.transform(df_train[StandardScale_cols])
    scaled_df_test[StandardScale_cols] = std_scaler.transform(df_test[StandardScale_cols])
    
    # Add the columns that weren't scaled.
    scaled_df_train[NoScale_cols_train] = df_train[NoScale_cols_train].copy()
    scaled_df_test[NoScale_cols_test] = df_test[NoScale_cols_test].copy()

    # Put the same order as before. 
    scaled_df_train = scaled_df_train[df_train.columns]
    scaled_df_test = scaled_df_test[df_test.columns]

    return scaled_df_train, scaled_df_test

In [9]:
def high_corr_filter(train : pd.DataFrame, 
                     test : pd.DataFrame, 
                     protected_cols = set(), 
                     thr = 0.99,
                     printing = False):
    """
    Gets rid of "duplicate columns" (ie: columns with a correlation higher than the treshold (thr))
    """

    # Select only numeric columns except protected ones
    features = [c for c in train.select_dtypes(include=[np.number]).columns
                if c not in protected_cols]

    to_drop = []

    # Only compute correlations if at least 2 numeric features exist
    if len(features) > 1:

        # Absolute correlation matrix
        corr = train[features].corr().abs()

        # Keep only upper triangle (avoid duplicate pairs) 
        # ---> this will put NaNs on the lower one ---> ignore warning for NaNs
        
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        
        # Find columns that have any correlation above threshold
        to_drop = [
            col for col in upper.columns
            if any(upper[col] > thr)
        ]

        # Print dropped columns
        if printing: 
            if to_drop:
                print("Dropping highly correlated columns (thr =", thr, "):")
                for col in to_drop:
                    print(" -", col)

        # Drop same columns in train & test
        train = train.drop(columns=to_drop, errors="ignore")
        test  = test.drop(columns=to_drop, errors="ignore")

    return train, test, to_drop

# 2) Data cleaning and analysis

Now that our functions are ready, we can apply them to both training and testing dataset. Then, we'll define some "modeling" functions to have a clearer structure in the modeling part of this project.

## 2.1) Loading training Data

In [10]:
# Load raw data
df_train = pd.read_csv('/kaggle/input/training/train (2).csv')

# Drop missing values -> 150 rows 
df_train = df_train.dropna()  

# Change names 
df_train = rename_columns(df_train) 

# Check impossible values 
df_train = check_impossible_values(df_train) 

# Create a new column
df_train = create_avg_temp_season_hour(df_train,['target_temp'], name='Bern', show_plot=True)  

# Encode hours and seasons
df_train = encode_hour_and_season(df_train) 

It seems to have created values correctly, high mean in summer and midday, low mean temperatures during nights and winter.

In [11]:
df_train.describe()

Given the large number of predictors, manually checking each column for anomalies is impractical. Since we have already verified the absence of impossible values, we instead rely plotting to understand the predictors. We assume that variables measuring the same phenomena exhibit similar behavior across cities. Based on this assumption, we visualize each predictor type using data from the city most comparable to Bern, namely Interlaken, as it is the nearest considering both altitude and distance.

In [12]:
# Plotting NOT scaled data distribution : Interlaken 

# Get columns where last 3 letters are "INT"
int_cols = [col for col in df_train.columns if col[-3:]=='INT'] # last 3 letters are "INT"
num_rows = 3
num_cols = 3

# Plot 9 figures
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 15)) # 9 plots 

for i, col in enumerate(int_cols):
    ax = axes[i // num_cols, i % num_cols]
    sns.histplot(df_train[col], kde=True,bins=70, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

Both wind measures are right skewed, which means that the mean is larger than the median. This reflects the fact that strong wind gusts or sustained high-wind periods are relatively rare events. The average solar radiation is an interesting one. Most values are close to 0 as a result of nights and cloudy days which displays this extremely skewed distribution with a long tail that depends on the sunny days. The two air pressure variables appear to follow approximately normal distributions with slight skewness. This behavior is expected, as air pressure is partly influenced by temperature, with colder air being denser and exerting greater pressure. Rain precipitation resembles an exponential distribution. As the measure is in mm/h, most will have a value close to 0 because Switzerland is not a heavy rain country. Sunshine distribution is slighlty different than solar radiation. Still, most values are close to 0 because on cloudy days and nights but then it is uniformly distributed until a slight peak at the end which represents summer days where a lot of days can have full sunshine hours. Air humidity is a left skewed distribution with a lot of values close to 100%. According to the website "Weather and Climate", the average yearly humidity in bern is around 77%. This graph may be overestimating values as Interlaken is between two lakes. Lastly, we can see a clear bimodal tendency in temperature, let's check that the targets also behave in this way. 

In [13]:
# Plotting the targets distribution (+Bern temperature now)

# Get columns with 'target' in the name.
int_cols = [col for col in df_train.columns if 'target' in col]
num_rows = 2
num_cols = 2

# Create 4 plots 
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 15))

for i, col in enumerate(int_cols):
    ax = axes[i // num_cols, i % num_cols]
    sns.histplot(df_train[col], kde=True,bins=70, ax=ax, color="red")
    ax.set_title(col)

plt.tight_layout()
plt.show()

As expected, temperature average follow a normal bimodal distribution. This is due the seasonality factor and can be seen well if we plot the seasons independently. 

In [14]:
# Recreate  a single column with the season name
season_cols = ["season_Winter", "season_Spring", "season_Summer", "season_Autumn"]
df_train["season"] = df_train[season_cols].idxmax(axis=1)

# Plot 
plt.figure(figsize=(12, 6))

sns.histplot(
    data=df_train,
    x="target_temp",      # Temperature column
    hue="season",         # Group by season
    kde=True,             # Smoothed curve
    bins=70,
    element="step",       # Outline look (cleaner)
    stat="density",       # Compare shape instead of counts
    common_norm=False     # Keeps each season curve independent
)

# Details
plt.title("Temperature Distribution by Season")
plt.xlabel("Temperature (°C)")
plt.ylabel("Density")
plt.show()

# Then we can drop this again...
df_train = df_train.drop('season', axis=1)

Every individual season follows a normal distribution. The addition of these seasons gives us a bimodal distribution with the peaks being the mean in winter and summer. 

## 2.2) Loading testing data

Now we can load the testing dataset and perform the same cleaning operations as our training. 

In [15]:
# How many missing values in the testing dataset ? 

df_raw_test = pd.read_csv('/kaggle/input/final-test/test.csv')
df_raw_test.isna().sum().sum()  

In [16]:
# Load raw data
df_test = pd.read_csv('/kaggle/input/final-test/test.csv')

# Change names 
df_test = rename_columns(df_test)  

# Check impossible values 
df_test = check_impossible_values(df_test) 

# Create new column 
df_test = create_avg_temp_season_hour(df_test,['target_temp'], name='Bern', show_plot=False)  

# Encode season and hour 
df_test = encode_hour_and_season(df_test) 

# Drop 'Id'... until final submission
df_final_test = df_test.drop('Id', axis=1) 

In [17]:
df_final_test.describe()

As we want to be sure that our 2 datasets match in terms of predictors, we can also plot the same predictors as before.

In [18]:
# Plotting NOT scaled data distribution : Interlaken 

# Get columns where last 3 letters are "INT"
int_cols = [col for col in df_final_test.columns if col[-3:]=='INT'] # last 3 letters are "INT"
num_rows = 3
num_cols = 3

# Plot 9 figures
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 15)) # 9 plots 

for i, col in enumerate(int_cols):
    ax = axes[i // num_cols, i % num_cols]
    sns.histplot(df_final_test[col], kde=True,bins=70, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

Everything is normal. 

In [19]:
df_train, df_final_test = scaling_data(df_train, df_final_test)

Now that we have scaled the data, we can re-plot our predictors to evaluate the scaling process and observe any changes or patterns in the data.


In [20]:
# Plotting scaled data. 

# Take columns where last 3 letters are "INT"
int_cols = [col for col in df_train.columns if col[-3:]=='INT']
num_rows = 3
num_cols = 3

# Make 9 plots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 15))  

for i, col in enumerate(int_cols):
    ax = axes[i // num_cols, i % num_cols]
    sns.histplot(df_train[col], kde=True,bins=70, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

Everything is scaled correctly with a mean of 0. However, we must keep in mind that this is not a min max scaler so some variables may have more values around 0 (average air pressure for example) while others have more values on the extremes (sum of sun time). This will make a difference in the coefficients for linear models.


In [21]:
# Drop columns correlated at 99% and more 

df_train ,df_final_test, dropped_cols = high_corr_filter(df_train, df_final_test, protected_cols=set(), thr=0.99, printing = True)

The only columns that were dropped were air pressure predictors which are obviously extremely correlated. After this last step, our final training dataset, "df_train", is of size : (7429, 91). While our testing dataset, "df_final_test", is of shape : (3247, 88), only the 3 target columns are missing.

## 2.3) Preparation : modeling functions

In [22]:
# Predictors : 
predictors_cols = list(df_train.columns)
predictors_cols.remove('target_temp_plus12')
predictors_cols.remove('target_temp_plus24')
predictors_cols.remove('target_temp_plus48')

# Targets : 
target_cols = ['target_temp_plus12','target_temp_plus24','target_temp_plus48']

# Splitting X and y.
X = df_train[predictors_cols]
y = df_train[target_cols]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Now we have (X,y) for Cross Validation and (X_train, y_train) for a normal fitting procedure.

We define 3 functions to help the clarity of our testing process: 
- getpred() : This function concatenates our predictions (3 columns) to the rest of predictions. The final structure of this DataFrame will be (number of models)x3 columns, 1486 rows.
- fit_and_cv() : This function  takes a model and uses MultiOutputRegressor to fit 3 different models to the 3 different targets using cross validation. This means that 15 fitting procedures are done everytime this functions is called.
- plot_results() : Plots 4 graph to see the behaviour of our predictions and residuals.

In [23]:
# We will store our predictions, results and model names in these.

result_error_df = pd.DataFrame(columns = ['model','Training_error 12h','Training_error 24h','Training_error 48h','Test_error 12h','Test_error 24h','Test_error 48h'])
predictions_df = pd.DataFrame()
models_done = []

In [24]:
def getpred(df : pd.DataFrame ,
            array : pd.Series,
            name : list) -> pd.DataFrame:
    """
    Concatenates the predictions (array) in the the predictions_df (df) as 3 columns(name_12,name_24 and name_48).
    """
    # Create a DataFrame from the array of prediction ---> structure : 3 columns, 7429 rows : 
    data_pred = pd.DataFrame(array, columns=name)
    
    df = df.reset_index(drop=True)
    data_pred = data_pred.reset_index(drop=True)

    # Concatenate the new predictions to the rest.
    df = pd.concat([df,data_pred], axis=1)
    return df

In [25]:
def fit_and_cv(model,
               X: pd.DataFrame,
               Y: pd.DataFrame,
               name: str,
               n_splits = 5) :

    # Access to outside variables from inside the function.
    global result_error_df, models_done, predictions_df
    
    # Add the model name to the list of models done. 
    models_done.append(name)

    # Create the base errors list to add to cv_result_error_df.
    result_error_list = [name, 10, 10, 10, 10, 10, 10]

    # Separate the folds
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Use MultiOutputRegressor
    model = MultiOutputRegressor(model, n_jobs=-1)

    # Store MAE per fold per target with size (n_splits, n_targets)
    n_targets = Y.shape[1]
    mae_train = np.zeros((n_splits, n_targets))
    mae_test = np.zeros((n_splits, n_targets))

    # Store predictions in an array of the same shape as the observed temperatures
    oof_predictions = np.zeros(y.shape)

    # Change the testing fold at every iteration.
    # Each train_idx and test_idx is a numpy array of row indices that we can use to create X_train and X_test.
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):

        # Create training and test set
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]

        # Fit model on train fold
        model.fit(X_train, Y_train)

        # Predict on train and test
        Y_pred_train = model.predict(X_train)
        Y_pred_test = model.predict(X_test)

        # Store predictions in original row order
        oof_predictions[test_idx, :] = Y_pred_test

        # Compute MAE per target for train and test 
        # And add them in the spot (fold,t) of the arrays we created before
        for t in range(n_targets):
            mae_train[fold, t] = mean_absolute_error(Y_train.iloc[:, t], Y_pred_train[:, t])
            mae_test[fold, t] = mean_absolute_error(Y_test.iloc[:, t], Y_pred_test[:, t])

    # Average across folds
    mean_mae_train = mae_train.mean(axis=0)
    mean_mae_test = mae_test.mean(axis=0)

    # Print the results 
    for i in range(n_targets):
        print(f"Target {i+1}")
        print(f"MAE CV Training {i+1}: {mean_mae_train[i]}")
        result_error_list[i+1] = round(mean_mae_train[i], 3)
        print(f"MAE CV Test {i+1}: {mean_mae_test[i]}")
        result_error_list[i+4] = round(mean_mae_test[i], 3)
        print()

    # Add the full results and prediction to the DataFrame 
    result_error_df.loc[len(result_error_df)] = result_error_list
    predictions_df  = getpred(predictions_df, oof_predictions, [f"{name}_12",f"{name}_24",f"{name}_48"])

    # Return model for graphs.
    return model


In [26]:
def plot_results(model_name: str,
                 horizon: int):
    """ 
    PLOT 1) Scatterplot: Observed vs Predicted
    PLOT 2) Residuals vs Observed temperature
    PLOT 3) Histogram of residuals
    PLOT 4) Boxplot: Observed vs Predicted
    """ 
    
    global y, predictions_df
    
    # Extract true and predicted values
    y_true = y[f"target_temp_plus{horizon}"].values
    y_pred = predictions_df[f"{model_name}_{horizon}"].values
    
    # Residuals
    residuals = y_true - y_pred
    mean_res = residuals.mean()
    std_res = residuals.std()

    # Shared limits for scatter
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    
    # Create figure with 2x2 subplots
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    
# - PLOT 1) -- Observed vs Predicted
    ax = axes[0, 0]
    ax.scatter(y_true, y_pred, alpha=0.4, s=10)
    ax.plot([min_val, max_val], [min_val, max_val],
            linestyle="--", linewidth=2, label="Perfect prediction")

    # Details
    ax.set_xlabel("Observed temperature")
    ax.set_ylabel("Predicted temperature")
    ax.set_title(f"{model_name} – Predicted vs Observed ({horizon}h)")
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    ax.set_aspect("equal", "box")
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend()

# - PLOT 2) -- Residuals vs Observed temperature + binned mean residuals
    ax = axes[0, 1]
    
    # Scatter of individual residuals
    ax.scatter(y_true, residuals, alpha=0.3, s=10, label="Residuals")
    
    # Zero line
    ax.axhline(0, linestyle="--", linewidth=2)
    
    # Binning 
    n_bins = 15
    bins = np.linspace(y_true.min(), y_true.max(), n_bins + 1)
    bin_indices = np.digitize(y_true, bins) - 1
    
    bin_centers = []
    mean_residuals = []
    
    for i in range(n_bins):
        mask = bin_indices == i
        if mask.sum() > 0:
            bin_centers.append((bins[i] + bins[i+1]) / 2)
            mean_residuals.append(residuals[mask].mean())
    
    bin_centers = np.array(bin_centers)
    mean_residuals = np.array(mean_residuals)
    
    # Plot binned mean residuals
    ax.plot(
        bin_centers,
        mean_residuals,
        marker="o",
        linewidth=2,
        color = 'red',
        label="Mean residual (binned)"
    )
    
    # Details
    ax.set_xlabel("Observed temperature")
    ax.set_ylabel("Residual (Observed − Predicted)")
    ax.set_title(f"{model_name} – Residuals vs Temperature ({horizon}h)")
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend()

# - PLOT 3) -- Histogram of residuals
    ax = axes[1, 0]
    ax.hist(residuals, bins=30, alpha=0.7, edgecolor="black")

    # Get a 0 vertical line, a mean line and 2 lines being mean+std.
    ax.axvline(0, linestyle="--", linewidth=2, label="Zero residual")
    ax.axvline(mean_res, linestyle=":", linewidth=2,
               label=f"Mean = {mean_res:.2f}")
    ax.axvline(mean_res + std_res, linestyle="--", color="red", linewidth=2, label=f"mean ±1 SD ({round(std_res,3)})")
    ax.axvline(mean_res - std_res, linestyle="--", color="red", linewidth=2)

    # Details
    ax.set_xlabel("Residuals (Observed − Predicted)")
    ax.set_ylabel("Frequency")
    ax.set_title(f"{model_name} – Residual distribution ({horizon}h)")
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(frameon=False)

# - PLOT 4) -- Boxplots
    ax = axes[1, 1]
    ax.boxplot(
        [y_true, y_pred],
        labels=["Observed", "Predicted"],
        patch_artist=True,
        boxprops=dict(facecolor="lightblue", alpha=0.7)
    )
    
    # Details
    ax.set_ylabel("Temperature")
    ax.set_title(f"{model_name} – Temperature distribution ({horizon}h)")
    ax.grid(True, linestyle=":", alpha=0.4)

    fig.tight_layout()
    plt.show()


Now we can go onto the fitting and testing of the models. 

# 3) Modeling 

For every model, we will do the following steps.
1) Tuning parameters : To tune model parameters, we have two main options: GridSearchCV and RandomizedSearchCV. Both work similarly but differ in the number of total model fits. GridSearchCV's total fits are (number of parameter permutation)x(folds). It exhaustively searches through all parameter combinations across all folds. On the other hand, RandomizedSearchCV's total fits are (n\_iter)x(folds) : for each iteration in n\_iter, it randomly picks a permutation and fits it to all folds. This method is preferred when the parameter grid is very large, but it is not necessarily faster for smaller grids. Because of the randomness, the results are not guaranteed to be globally optimal. Realistically, we have 4 models (tree based models) that require a higher level of parameter tuning. Ideally, with sufficient computational power and time, we should do a GridSearch for every target and for a fairly large grid of parameters. However, the Kaggle notebook environment limits such extensive computation. In the code, comments cells with --CODE-- and --RESULT-- represent a GridSearch that has already been performed. As it is said in the XGboost documentation : ``Parameter tuning is a dark art in machine learning, the optimal parameters of a model can depend on many scenarios. So it is impossible to create a comprehensive guide for doing so.''
2) Training and testing : We will use fit_and_cv() to get results and add them to our general DataFrame.
3) Understand important features of the model : What predictors or parameters are important in the model ?
4) Use plots to understand better why some models predict better than others or behave differently.

## 3.1) Linear Regression

Linear regression is an extremely simple model, it will search for three coefficients per predictors, one for every target and three intercept values. There aren't any tuning paramteres in this model. Linear Regression is the least complex of all the models discussed in this report, yet it serves a critical role as a baseline, a starting point. By establishing the performance of a simple linear approach, we create a benchmark against which more complex models can be evaluated.

In [27]:
# Model
lin_reg = LinearRegression()

# Fit
lin_reg = fit_and_cv(lin_reg, X,y, name='Lin_reg')

Interestingly, the best performance occurs when predicting 24 hours ahead. Temperature does not have a perfectly linear relationship with the predictors overall, but the 24-hour forecast aligns with the same time of day, making the relationship effectively more linear and therefore easier for the model to capture. We can see that there isn't much overfitting. Linear regression is a good baseline due to the smooth and persistent nature of temperature.

In [28]:
# Coefficients weight graph for linear regression: all horizons

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

# Loop over the three targets
for i, ax in enumerate(axes):

    # Create a DataFrame with the features and the coefficients accordingly 
    results = pd.DataFrame({
        'feature': list(X.columns),
        'weight': lin_reg.estimators_[i].coef_
    })

    # Sort to get the most extreme ones (both positive and negative)
    sorted_results = results.sort_values('weight', ascending=False)
    top_5 = sorted_results.head(5)
    tail_5 = sorted_results.tail(5)
    important_results = pd.concat([top_5, tail_5], axis=0)

    # Plot
    sns.barplot(
        data=important_results,
        x='weight',
        y='feature',
        ax=ax,
        palette='pink'
    )

    ax.set_title(f"Highest coefficients in Linear Regression – Target {i+1}")
    ax.set_xlabel("Coefficient value")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


 The coefficients ($\beta_{j,h}$) give us a sense of how predictors relate to Bern. For every change in 1 unit of the predictor then the output value changes by the coefficients value. Even with scaled predictors, coefficients aren't a perfect measure of feature importance. This is because it doesn't take into consideration the distribution of the values. It is rarer for a low variance variable like average pressure to change drastically because of its distribution. So when it does happen, it has to have a bigger impact, which means a higher coefficient.
 
 We can try a sort of predictors selection as too many "useless" predictors can hurt the model. The next function iterates through all predictors and adds to the regression the one that lowers MAE the most. It stops when no predictors help the accuracy. This can be a good feature importance measure. 

In [29]:
def predictors_iteration():
    """
    This function goes through all predictors and adds to the regression the one that lowers MAE the most to find the optimal 
    number of predictors.
    Returns 4 lists : one for each time target, and 1 for commmon predictors.
    """
    global X_train, X_test, y_train, y_test, target_cols

    # This list will contain 3 lists, each for a different time target.
    selected_features_all = []

    # For every target : we create a "selected_features" list in which we append the added predictors.
    for target in target_cols:
        selected_features = []

        # Remaining features to go through 
        remaining_features = list(X_train.columns)

        # Set a first MAE
        best_mae = 10000

        # In case we want to impose a max.
        max_features = len(X_train.columns)

        
        for i in range(max_features):
            improvement = False
            best_feature = None

            # Loop through the features 
            for feature in remaining_features:
                current_features = selected_features + [feature]

                # Train a linear regression with that feature.
                model = LinearRegression()
                model.fit(X_train[current_features], y_train[target])
                y_pred = model.predict(X_test[current_features])
                mae = mean_absolute_error(y_test[target], y_pred)

                # If MAE is lower, we replace it, and put the current feature = best feature
                if mae < best_mae:
                    best_mae = mae
                    best_feature = feature
                    improvement = True

            # At the end of the loop through the features we add the best,
            # and remove it from the remaining features to loop through 
            if improvement:
                selected_features.append(best_feature)
                remaining_features.remove(best_feature)
            
            else:
                # No further improvement -> we break the loop.
                break

        # We print the number of features and the MAE for each time target.
        print(f"For target: {target}, number of predictors:{len(selected_features)}, MAE: {best_mae:.4f}") 

        # Add the list of selected features.
        selected_features_all.append(selected_features)

    # This list will contain the predictors that are common to the three time targets.
    feat_all_3 = []
    for feat in selected_features_all[0]:
        if feat in selected_features_all[1]: 
            if feat in selected_features_all[2]:
                feat_all_3.append(feat)
    
    return selected_features_all[0], selected_features_all[1], selected_features_all[2], feat_all_3
selected_feat12,selected_feat24,selected_feat48, common_feat = predictors_iteration()
print()
print('The common feat for all 3 horizons are : ', common_feat)

First of all, the further we predict into the future, the less predictors the model needs. This is because the further we predict into the future the less changes in temperature are due to our data in this moment in time. Basically we lose relationship to the target, that means that our model's best chance at predicting the temperature is to forecast a value close to the average. It can be seen as predicting a trend in a time series because cyclical components are hard to predict. 

Also, the best prediction is still for the 24h forecast for the same reasons mentioned above. If we consider important features the ones that have been selected in all 3 horizon forecasts, the common features seem much more logical to us : 6 temperature columns, and 1 hour column.

In [30]:
plot_results('Lin_reg', 12)

In [31]:
# Let's look at the results:
plot_results('Lin_reg', 24)

In [32]:
plot_results('Lin_reg', 48)

Even though the standard deviation is high, a good sign of stable prediction is that the residuals are normally distributed around 0 because if the trend is predicted rightly the residuals should white noise (normally distributed N(0,1)). The boxplot of predicted temperatures is a little shrunk towards the center because it predicts closer to the mean to minimze errors. This factor can be seen in the evolution of the "residuals vs temperatures" graph because cold temperatures are predicted too hot and viceversa. In the first scatter plot, we can see that by increasing the predicting time period, the model predicts "safer" values closer to the mean.

## 3.2) Ridge Regression

We tried to manually select features linear regression part. Now we can try 2 models that use regularization to provide variable selection: Ridge and Lasso. In Ridge, we only tune $\alpha$, which penalizes the model and shrinks coefficients to minimize the $RSS_{Ridge}$. We can look for the best alpha for each of the targets.

In [33]:
# Alpha grid (shared for all targets)
ridge_reg_grid = 10 ** np.linspace(-2, 6, num=30)

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, t in zip(axes, target_cols):

    # Pipeline
    ridge_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(fit_intercept=True))
    ])

    # Cross-validation
    folds = KFold(n_splits=10, shuffle=True, random_state=42)

    ridgeCV = GridSearchCV(
        estimator=ridge_pipe,
        param_grid={"ridge__alpha": ridge_reg_grid},
        scoring="neg_mean_absolute_error",
        cv=folds
    )

    ridgeCV.fit(X, y[t])

    # Extract CV results
    mean_scores = -ridgeCV.cv_results_["mean_test_score"]
    se_scores = ridgeCV.cv_results_["std_test_score"] / np.sqrt(folds.get_n_splits())
    alphas = ridgeCV.cv_results_["param_ridge__alpha"].astype(float)

    # Best alpha and 1-SE rule
    best_idx = np.argmin(mean_scores)
    min_alpha = alphas[best_idx]
    threshold = mean_scores[best_idx] + se_scores[best_idx]
    one_se_alpha = np.max(alphas[mean_scores <= threshold])

    # ---- Plot ----
    ax.errorbar(alphas, mean_scores, yerr=se_scores, fmt='o', capsize=3)
    ax.axvline(min_alpha, ls="dotted", color="grey", label="Min MAE")
    ax.axvline(one_se_alpha, ls="dotted", color="black", label="1-SE rule")
    ax.axhline(threshold, ls="dotted", color="grey")

    ax.set_xscale("log")
    ax.set_title(f"Target: {t}")
    ax.set_xlabel("alpha")
    ax.grid(True)

    # Print results
    print(f"For target: {t}")
    print("Minimum alpha:", min_alpha)
    print("1-SE alpha:", one_se_alpha)
    print("Best MAE:", mean_scores[best_idx])
    print()

# Shared y-label
axes[0].set_ylabel("Mean Absolute Error")

plt.suptitle("Ridge Regression CV Error (per target)", fontsize=14)
plt.tight_layout()
plt.show()


Like we saw in the results of the "predictors_iteration()", the more we predict into the future the more regularization the model needs because it tries to predict something closer to the mean temperature, which is mainly explained by other temperatures. To confirm this we'll have to look at a feature importance measure.

Let's see the results for a general alpha that considers the three targets at once. 

In [34]:
# Find parameters for ridge.
ridge_reg_grid = 10 ** np.linspace(-2, 6, num = 30) # -> [0.01, 0.0189, 0.0357, ..., 316227.76, 575439.94, 1000000] as alpha 

# Create a Pipeline : Scaling a value a second time doesn't change because (x-0)/1=x !
ridge_reg = Ridge(alpha = 0.9, fit_intercept = True)
ridge_reg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge_reg', ridge_reg)
])

# GridSearch
folds = KFold(n_splits = 10, random_state = 42, shuffle = True)
ridge_reg_params = {"ridge_reg__alpha" : ridge_reg_grid}
ridgeCV = GridSearchCV(
    estimator = ridge_reg_pipe,
    param_grid = ridge_reg_params,
    scoring = "neg_mean_absolute_error",
    cv = folds
)

ridgeCV.fit(X, y)
pd.DataFrame(ridgeCV.cv_results_)

mean_scores = -ridgeCV.cv_results_["mean_test_score"]
se_scores = ridgeCV.cv_results_["std_test_score"] / np.sqrt(ridgeCV.n_splits_)
alphas = ridgeCV.cv_results_["param_ridge_reg__alpha"].data

best_index = np.argmin(mean_scores)
min_alpha_ridge = alphas[best_index]
threshold_ridge = mean_scores[best_index] + se_scores[best_index]
one_se_rule_alpha_ridge = np.max(alphas[mean_scores <= threshold_ridge])
print("For all targets")
print("Minimum alpha:", min_alpha_ridge)
print("1-SD alpha:", one_se_rule_alpha_ridge)
print("Best score for ridge:", (np.min(mean_scores)))
print()

The alpha is a tradeoff between the targets. As we saw, the results for Ridge are barely different than the linear regression so we will keep this "general" alpha for the rest keeping in mind that it's slightly worse.

In [35]:
# Plotting the error-alpha graphic.
plt.errorbar(x=ridge_reg_grid, y=mean_scores, yerr = se_scores, fmt='o', capsize=3)

# Add 2 vertical lines : one for the minimum error alpha, another for the alpha following the one SE rule.
plt.axvline(min_alpha_ridge, ls='dotted', color="grey") 
plt.axvline(one_se_rule_alpha_ridge, ls='dotted', color="grey")

# Add a horizontal line for the threshold (ie. 1 SE over min alpha)
plt.axhline(threshold_ridge, ls='dotted', color="grey")

# Details
plt.title("Ridge regressor CV error across the 3 targets")
plt.xlabel('log(alpha)')
plt.ylabel('Mean Absolute Error')
plt.xscale('log')
plt.ylim(2.175,2.3)
plt.show()

To check for important features, we can look at the least shrunken features when alpha is high. This means that with a high regularization, what features are gonna impact the most when they change. 

In [36]:
# choose target 0, 1, or 2
target_id = 1  

coefs = []

for alpha in ridge_reg_grid:
    ridge_reg_pipe.set_params(ridge_reg__alpha=alpha)
    ridge_reg_pipe.fit(X_train, y_train)
    coefs.append(ridge_reg_pipe.named_steps['ridge_reg'].coef_[target_id])

# Shape = (n_alphas, n_features)
coefs = np.array(coefs) 

# Pick the highest alpha = strongest shrinkage -> last row
coefs_strong = np.abs(coefs[-1])  

# Choose top 10 predictors
k = 10  
top_idx = np.argsort(coefs_strong)[-k:]  

# Create plot
plt.figure(figsize=(10,6))

# Plot with lightgray 'unimportant' features
for i in range(coefs.shape[1]):
    plt.plot(ridge_reg_grid, coefs[:, i], color='lightgray', linewidth=1)

# Plot only most important ones with strong color & legend
for i in top_idx:
    plt.plot(ridge_reg_grid, coefs[:, i], linewidth=2, label=X.columns[i])

# Details 
plt.xscale('log')
plt.xlabel("$\\alpha$", fontsize=14)
plt.ylabel("$\\beta_i$", fontsize=14)
plt.title(f"Ridge Coefficient Paths — Target 24h")
plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1), title="Least Shrunken Predictors when large $\\alpha$")
plt.tight_layout()
plt.show()

Similarly to a linear regression, the most important features for prediction are mean temperatures across the cities. On the other hand, high starting coefficients, that we saw in the linear regression coefficients are air pressure variables, are shrunken very rapidly.

In [37]:
# Model
ridge = Ridge(alpha = one_se_rule_alpha_ridge, fit_intercept = True)

# Fit
ridge = fit_and_cv(ridge, X,y, name='Ridge')

 Results are a little worse than linear regression for the general alpha. No actual benefit from regularization... Even when alpha is chosen individually for each target the results are marginally better because in both cases we only captures linear relationships. 

In [38]:
# Coefficients weight graph for Ridge: all horizons

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

# Loop over the three targets
for i, ax in enumerate(axes):

    # Create a DataFrame with the features and the coefficients accordingly 
    results = pd.DataFrame({
        'feature': list(X.columns),
        'weight': ridge.estimators_[i].coef_
    })

    # Sort to get the most extreme ones (both positive and negative)
    sorted_results = results.sort_values('weight', ascending=False)
    top_5 = sorted_results.head(5)
    tail_5 = sorted_results.tail(5)
    important_results = pd.concat([top_5, tail_5], axis=0)

    # Plot
    sns.barplot(
        data=important_results,
        x='weight',
        y='feature',
        ax=ax,
        palette='Greens'
    )

    ax.set_title(f"Highest coefficients in Ridge – Target {i+1}")
    ax.set_xlabel("Coefficient value")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


Not really an accurate measure to imply what features are the most important. In the first target, seasons are a big factor because it won't change in 12h.  On the other hand, there are more temperature variables in the two further targets.

In [39]:
plot_results('Ridge', 12)

In [40]:
plot_results('Ridge', 24)

In [41]:
plot_results('Ridge', 48)

Very similar behaviour to linear regression because our alpha is small so our model is close to a simple linear regression approach. 

## 3.3) Lasso Regression

Like Ridge regression, Lasso also introduces a regularization parameter 𝛼 to penalize model complexity. However, while Ridge regression shrinks coefficients continuously toward zero, Lasso is able to shrink some coefficients exactly to zero, effectively performing variable selection. Although this property may appear advantageous, the results obtained with both linear and Ridge regression suggest that regularization does not substantially improve predictive performance in this context. This indicates that the underlying relationship between the predictors and the target variable is likely non-linear. Consequently, a linear model with L1 regularization is not expected to yield strong predictive results for this problem.

In [42]:
# Alpha grid (shared for all targets)
lasso_grid = 10 ** np.linspace(-2, 6, num=30)

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, t in zip(axes, target_cols):

    # Pipeline
    lasso_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(fit_intercept=True))
    ])

    # Cross-validation
    folds = KFold(n_splits=10, shuffle=True, random_state=42)

    lassoCV = GridSearchCV(
        estimator=lasso_pipe,
        param_grid={"lasso__alpha": lasso_grid},
        scoring="neg_mean_absolute_error",
        cv=folds
    )

    lassoCV.fit(X, y[t])

    # Extract CV results
    mean_scores = -lassoCV.cv_results_["mean_test_score"]
    se_scores = lassoCV.cv_results_["std_test_score"] / np.sqrt(folds.get_n_splits())
    alphas = lassoCV.cv_results_["param_lasso__alpha"].astype(float)

    # Best alpha and 1-SE rule
    best_idx = np.argmin(mean_scores)
    min_alpha = alphas[best_idx]
    threshold = mean_scores[best_idx] + se_scores[best_idx]
    one_se_alpha = np.max(alphas[mean_scores <= threshold])

    # ---- Plot ----
    ax.errorbar(alphas, mean_scores, yerr=se_scores, fmt='o', capsize=3)
    ax.axvline(min_alpha, ls="dotted", color="grey", label="Min MAE")
    ax.axvline(one_se_alpha, ls="dotted", color="black", label="1-SE rule")
    ax.axhline(threshold, ls="dotted", color="grey")

    ax.set_xscale("log")
    ax.set_title(f"Target: {t}")
    ax.set_xlabel("log(alpha)")
    ax.grid(True)

    # Print results
    print(f"For target: {t}")
    print("Minimum alpha:", min_alpha)
    print("1-SE alpha:", one_se_alpha)
    print("Best MAE:", mean_scores[best_idx])
    print()

# Shared y-label
axes[0].set_ylabel("Mean Absolute Error")

plt.suptitle("Lasso Regression CV Error (per target)", fontsize=14)
plt.tight_layout()
plt.show()


Error goes up extremely quickly when using L1 regularization. The pattern is still clear : further prediction equals more regularization to predict closer to the mean.

Let's see the results for a general alpha that considers the three targets at once. This time we also change the grid values to check into a more accurate grid space.

In [43]:
lasso_grid = 10**np.linspace(-2, 2, num=30)
lasso = Lasso(fit_intercept = True)
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', lasso)
])

params_lasso = {"lasso__alpha" : lasso_grid}
lassoCV = GridSearchCV(
    estimator = lasso_pipe,
    param_grid = params_lasso,
    scoring = "neg_mean_absolute_error",
    cv = folds
)
lassoCV.fit(X, y)

# Turn into a DataFrame
pd.DataFrame(lassoCV.cv_results_)

# Get the scores and alphas 
mean_scores = -lassoCV.cv_results_["mean_test_score"]
se_scores = lassoCV.cv_results_["std_test_score"] / np.sqrt(lassoCV.n_splits_)
alphas = lassoCV.cv_results_["param_lasso__alpha"].data


best_index = np.argmin(mean_scores)
min_alpha_lasso = alphas[best_index]

threshold_lasso = mean_scores[best_index] + se_scores[best_index]
one_se_rule_alpha_lasso = np.max(alphas[mean_scores <= threshold_lasso])

print("Minimum alpha:", min_alpha_lasso)
print("1-SD alpha:", one_se_rule_alpha_lasso)
print("Best score for lasso:", (np.min(mean_scores)))

In [44]:
# Plotting the error-alpha graphic.
plt.errorbar(x=lasso_grid, y=mean_scores, yerr = se_scores, fmt='o', capsize=3)

# Add 2 vertical lines : one for the minimum error alpha, another for the alpha following the one SE rule.
plt.axvline(min_alpha_lasso, ls='dotted', color="grey")
plt.axvline(one_se_rule_alpha_lasso, ls='dotted', color="grey")

# Add a horizontal line for the threshold (ie. 1 SE over min alpha)
plt.axhline(threshold_lasso, ls='dotted', color="grey")

# Details
plt.title("Lasso regressor CV error")
plt.xlabel('log(alpha)')
plt.ylabel('Mean Absolute Error')
plt.xscale('log')
plt.show()

When minimizing the mean MAE across targets regularization is small even with a smaller grid...

In [45]:
# Choose target 0, 1, or 2
target_id = 1  

coefs = []

for alpha in ridge_reg_grid:
    lasso_pipe.set_params(lasso__alpha=alpha)
    lasso_pipe.fit(X_train, y_train)
    coefs.append(lasso_pipe.named_steps['lasso'].coef_[target_id])

# coefs : shape (n_alphas, n_features)
coefs = np.array(coefs)  

# Pick the highest alpha = strongest shrinkage -> last row
coefs_strong = np.abs(coefs[-1]) 

# Pick top k most important
k = 10  
top_idx = np.argsort(coefs_strong)[-k:] 

plt.figure(figsize=(10,6))

# Plot all features in light color
for i in range(coefs.shape[1]):
    plt.plot(lasso_grid, coefs[:, i], color='lightgray', linewidth=1)

# Plot only most important ones with strong color & legend
for i in top_idx:
    plt.plot(lasso_grid, coefs[:, i], linewidth=2, label=X.columns[i])

# Details
plt.xscale('log')
plt.xlabel("$\\alpha$", fontsize=14)
plt.ylabel("$\\beta_i$", fontsize=14)
plt.title(f"Ridge Coefficient Paths — Target {target_id + 1}")
plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1), title="Least Shrunken Predictors")
plt.tight_layout()
plt.show()

 Looking at the coefficients for high alpha is not a good move for lasso because it actually shrinks them to 0. In this case, looking at the coefficients for our optimal alpha might be better to interpret feature importance.

In [46]:
# Lasso Regression 
lasso = Lasso(alpha =  one_se_rule_alpha_lasso, fit_intercept = True)

# Fit
lasso = fit_and_cv(lasso, X,y, name='Lasso')

 As expected, the results are not good, which confirms that a linear model, with or without regularization, doesn't seem to be the right fit for this problem. Note that the model doesn't allow for a high alpha so no features are actually cut off.

In [47]:
# Coefficients weight graph for Lasso: all horizons

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

# Loop over the three targets
for i, ax in enumerate(axes):

    # Create a DataFrame with the features and the coefficients accordingly 
    results = pd.DataFrame({
        'feature': list(X.columns),
        'weight': lasso.estimators_[i].coef_
    })

    # Sort to get the most extreme ones (both positive and negative)
    sorted_results = results.sort_values('weight', ascending=False)
    top_5 = sorted_results.head(5)
    tail_5 = sorted_results.tail(5)
    important_results = pd.concat([top_5, tail_5], axis=0)

    # Plot
    sns.barplot(
        data=important_results,
        x='weight',
        y='feature',
        ax=ax,
        palette='coolwarm'
    )

    ax.set_title(f"Highest coefficients in Lasso – Target {i+1}")
    ax.set_xlabel("Coefficient value")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


 These results seem more in lign with our logic. First target's most important features are the hours, because of the night / day swith. The two other targets focus increasingly more on temperatures, like before.

In [48]:
plot_results('Lasso', 12)

In [49]:
plot_results('Lasso', 24)

In [50]:
plot_results('Lasso', 48)

We observe a slightly higher standard deviation in the residuals as the regularization becomes harsher. As we suggested, regularization does not help the model and instead leads to more averaged predictions. As a result, part of the true relationship is not captured by the model, which increases the spread of the residuals.

## 3.4) KNN 

KNN is a purely geometrical model that avrages the values of the neighbours to get an output. We don't expect this have good results mainly because of the number of predictors that we have. This means that our model must look for neigbhours in a 91 dimension space. At this level, actual distances become less meaningful. Also, Knn will also have a "smooth" predictions problem because the model won't predict something outside of an average from the training data.

The only tuning parameter we search for is the number of neighbours, k. With an increasing k, the predictions become smoother as we take the average of more neighbours.

In [51]:
# Find the best parameters for KNN
for t in target_cols:
    
    # Model 
    knn = KNeighborsRegressor()

    folds = KFold(n_splits=10, shuffle=True, random_state=1)

    # Parameters Grid 
    hyper_parameters = {"n_neighbors": np.arange(1, 20, 2)}

    knn_CV = GridSearchCV(
        estimator=knn,
        param_grid=hyper_parameters,
        scoring="neg_mean_absolute_error",
        cv=folds
    )

    knn_CV.fit(X, y[t])
    
    # Compute "best" k value and its index:
    resCV = knn_CV.cv_results_

    test_MAEs = -resCV["mean_test_score"]
    se_test_MAEs = resCV["std_test_score"] / np.sqrt(10)
    k_grid = resCV["param_n_neighbors"].data
    
    # index of the k value with the lowest MAE estimate
    index_best = np.argmin(test_MAEs)
    best_k = k_grid[index_best]

    print(f"Optimal number of neighbours for target {t}: {best_k}")


In line with our points made from linear models. Further targets require higher number of neighbours to smooth out predictions.

In [52]:
# Find the best parameters for KNN.

# Model 
knn = KNeighborsRegressor()

folds = KFold(n_splits=10 , shuffle=True, random_state=1)

# Parameters Grid 
hyper_parameters = {"n_neighbors" : np.arange(1,20,2)}
knn_CV = GridSearchCV(estimator=knn,
                      param_grid=hyper_parameters,
                      scoring="neg_mean_absolute_error",
                      cv=KFold(n_splits=10 , shuffle=True, random_state=1))

knn_CV.fit(X, y)

# Compute "best" k value and its index:
resCV = knn_CV.cv_results_
pd.DataFrame(resCV)

test_MAEs = -resCV["mean_test_score"]
se_test_MAEs = resCV["std_test_score"] / np.sqrt(10)
k_grid = resCV["param_n_neighbors"].data

# index of the k value with the lowest MAE estimate
index_best = np.argmin(test_MAEs) 
best_k = k_grid[index_best]
print(f"Optimal number of neighbours : {best_k}")

In [53]:
se_test_MAEs = resCV['std_test_score']/np.sqrt(knn_CV.n_splits_)
threshold = test_MAEs[index_best] + se_test_MAEs[index_best]
candidates = k_grid[test_MAEs <= threshold]
one_se_rule_best_k = np.max(candidates)

# Plot an errorbar for KNN:
plt.figure(figsize=(10,6))
plt.errorbar(x=k_grid, y=test_MAEs, yerr=se_test_MAEs, fmt='.', capsize=3)

# 2 Vertical lines : one at the k yielding minimum CV MAE, one at best k value according to 1 std err rule
plt.axvline(best_k, ls='dotted', color="grey") 
plt.axvline(one_se_rule_best_k, ls='dotted', color="grey")

 # 1 horizontal line : threshold for 1 std err rule
plt.axhline(test_MAEs[index_best] + se_test_MAEs[index_best], ls='dotted', color="grey")

# Details
plt.title("kNN regressor CV error")
plt.xlabel('k (nb neighbors)')
plt.ylabel('Mean Absolute Error')
plt.show()

In [54]:
# Model
knn = KNeighborsRegressor(n_neighbors=best_k)

# Fit 
knn = fit_and_cv(knn, X,y, name='KNN')

 The results are worse than linear models. Knn suffers from the curse of dimensionality and a problem to predict extremes. 

In [55]:
# Plot results.
plot_results('KNN', 24)

We can clearly see the in the boxplot how KNN shrinks predictions. Also, the residuals mean shown in the histogram is a lot further from 0 than previous models.

## 3.5) Decision Tree 

The base decision tree will not be a good model because it overfits extremely easily and is just not complex enough. However it will be a good base model for bagging, boosting and random forests. 

There are a lot of tuning parameters in tree models. We only choose to tune some of them for timely and computational reasons. **"max_depth"** controls how many splits the tree is allowed to make from root to leaf, it is basically a complexity measure. A small depth means a short tree that underfits the data and has high bias. Whereas a high tree depth can overfit the data and have a high variance. **"min_sample_split"** controls the minimum number of observations needed in a node to allow a split. If this measure is too high the tree will only split big branches and have smoother predictions (bigger leaves) whereas if the measure is low, it can split too easily and overfit. **"min_samples_leaf"** controls the minimum number of observations in each terminal node (ie. the size of the leafs). Big leaf size means an average prediction over more samples and a small leaf size means a higher risk of overfitting.

In [56]:
# Find the best parameters for DecisionTree.

def find_params_tree(X,y):
    tree = DecisionTreeRegressor(random_state=42)
    model = MultiOutputRegressor(tree)
    
    # Define hyperparameter grid
    param_grid = {
        'estimator__max_depth': [None, 5,7, 10, 15],
        'estimator__min_samples_split': [2, 5, 10, 20],
        'estimator__min_samples_leaf': [4, 8, 10, 12],
    }
    
    # GridSearch
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=0 
    )
    
    grid.fit(X, y)
    
    print("Best params:", grid.best_params_)
    print("Best score  (MAE):", -grid.best_score_)

find_params_tree(X,y)

In [57]:
# Showcasing tree depth and error relationship

# Inizialize lists for training and test errors of all trees. This will be a list of lists with the errors for every target.
training_errors, test_errors = [],[]

# For all different depth :  we create a Decision Tree and append the training and test errors to our list. 
for i in range(1,26,2):

    #  Create a Decision Tree. Fit and predict
    model_tree = DecisionTreeRegressor(max_depth=i,min_samples_leaf= 8, min_samples_split= 20)  
    model_tree.fit(X_train,y_train)
    y_pred = model_tree.predict(X_test)
    y_train_pred = model_tree.predict(X_train)

    # Inizialize the errors of a single tree
    error_single_tree=[]
    test_error_single_tree = []

    # Get the results for the 3 targets 
    for j in range(y.shape[1]):
        mae_test = mean_absolute_error(y_test.iloc[:, j], y_pred[:, j])
        mae_train = mean_absolute_error(y_train.iloc[:, j], y_train_pred[:, j])

        # Add to the list
        error_single_tree.append(mae_train)
        test_error_single_tree.append(mae_test)

    # Add the single tree to the list for all trees
    training_errors.append(error_single_tree)
    test_errors.append(test_error_single_tree)

# Create a DataFrae for the full results
decision_tree_results_df = pd.DataFrame(columns = ['Training_error 12h','Training_error 24h','Training_error 48h','Test_error 12h','Test_error 24h','Test_error 48h'])

# Iteratively create new rows of errors per tree.
for i in range(len(training_errors)):
    new_row = [training_errors[i][0],training_errors[i][1],training_errors[i][2],test_errors[i][0],test_errors[i][1],test_errors[i][2]]
    decision_tree_results_df.loc[len(decision_tree_results_df)] = new_row

# Add a depth column 
decision_tree_results_df['Depth'] = range(1,26,2)

# Plot by adding a new target in each iteration. 
plt.figure(figsize=(8,5))
for i in range(3):
    plt.plot(decision_tree_results_df['Depth'], decision_tree_results_df.iloc[:,i], marker='o', label=decision_tree_results_df.columns[i])
    plt.plot(decision_tree_results_df['Depth'], decision_tree_results_df.iloc[:,i+3], marker='x', label=decision_tree_results_df.columns[i+3])

plt.legend()
plt.show()

In [58]:
# Tree model
tree = DecisionTreeRegressor(max_depth= 7, min_samples_leaf= 8, min_samples_split= 20)

# Fit
tree = fit_and_cv(tree, X,y, name='Tree')

A single tree is a very weak model but the results of it are close to a full KNN model. We can expect tree models, like bagging, boosting and random forests, to perform well in this problem because they capture more relations than linear models and knn. 

In [59]:
# This model is only a base model. We'll only plot for 24h.
plot_results('Tree', 24)

We can see the very clear leaves structure in the graphs. The residuals' mean is close to 0 but the standard deviation is too high. A bagging model, should get good results by lower the variance of individual trees.

## 3.6) Tree Bagging  

Bagging models reduce variance by combining many high variance models, like decision trees. It basically draws multiple bootstrap samples from the training data, trains a tree on each sample and aggregate predictions by averaging. Each tree sees a slightly different version of the dataset, which makes their errors less correlated.

The first parameter that we will tune in this model in **'n_estimators'**, which controls the number of trees that are grown (= number of bootstrap models). Increasing the number of trees improves stability up to a point where accuracy stagnates. **"max_sample"** is the fraction of sample drawn for each tree. If samples are larger then trees will be more similar because they'll see bigger parts of each others data. However too little data per tree will increase bias. **"min_samples_leaf"** refers to the same base tree parameter as before that controls the size of the leaves. 

In [60]:
# Find bagging parameters

def find_params_bag(X,y):
    
    # Model 
    bag = RandomForestRegressor(max_features=None, n_jobs=-1, random_state=42)  
    model = MultiOutputRegressor(bag)
    
    # Parameters grid
    param_grid = {
        'estimator__n_estimators': [300, 400, 500],
        'estimator__min_samples_leaf': [1,2, 4],
        'estimator__max_samples': [0.8,0.9,1.00]
    }

    # Grid Search
    grid_search = GridSearchCV(
        model,
        param_grid,
        cv=5,
        scoring="neg_mean_absolute_error",
        random_state=42,
        n_jobs=-1
    )
    grid_search.fit(X, y)
    
    
    print("Best params:", grid.best_params_)
    print("Best score  (MAE):", -grid.best_score_)

# -- CODE -- 
# find_params_bag(X,y)

# -- RESULT -- 
# Best parameters found: {'max_samples': 1.0, 'min_samples_split': 2, 'n_estimators': 400}



In [61]:
# Bagging decision tree. 
bag = RandomForestRegressor(max_features=None,
                            min_samples_split=2,
                            n_estimators=400,
                            max_samples=1.0,
                            n_jobs=-1,
                            random_state=42
                           )  

# Fit 
bag = fit_and_cv(bag, X,y, name='BagTree')

Obviously, the error still increases over time. We can see that results are far better than other models. There is a strong overfitting behaviour.

In [62]:
# Feature importances for bagging: all horizons

# Create 3 plots into 1.
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

# Loop over the three targets
for i, ax in enumerate(axes):

    # Get the right model
    bag_ = bag.estimators_[i] 

    # Get the mean importance from all the trees
    importances = np.mean([est.feature_importances_ for est in bag_.estimators_], axis=0)  

    # Create a DataFrame with the features and the coefficients accordingly 
    results = pd.DataFrame({
        'feature': list(X.columns),
        'weight': importances
    })

    # Sort to get the most extreme ones (both positive and negative)
    sorted_results = results.sort_values('weight', ascending=False)
    top_10 = sorted_results.head(10)
    important_results = pd.concat([top_10],axis =0)

    # Plot
    sns.barplot(
        data=important_results,
        x='weight',
        y='feature',
        ax=ax,
        palette='viridis'
    )

    # Details
    ax.set_title(f"Feature importance in Bagging – Target {i+1}")
    ax.set_xlabel("Coefficient value")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


Feature importance for tree models shows clearly what features the model relies on the most. The most important predictors by a large margin are the average temperatures. The results are normal and confirm a good behaviour of the model. 

In [63]:
# Plot results 
plot_results('BagTree', 12)

In [64]:
plot_results('BagTree', 24)

In [65]:
plot_results('BagTree', 48)

Residuals' mean is 0 and standard deviation went down a lot from linear models. Looking at the boxplots and scatterplots, we can see that the model still shrinks predictions towards the center. Tree models can clearly capture the true relations better.Mean residuals line in the scatter plot shows the evolution of safer predictions by showcasing an increasing mean on the extremes. 

## 3.7) Tree Boosting

Gradient Boosting is an ensemble method that builds models sequentially, where each new tree is trained to correct the errors made by the previous ones. So it starts with a simple tree then builds a tree to fit the residuals, updates the residuals and sequentially builds trees and adds a shrunken version of those trees to the final model. Unlike bagging, it actively reduces bias.

The principal parameters for gradient boosting are : **"n_estimators"**, **"learning_rate"**, **"max_depth"** and **"subsample"**.Like bagging, the first parameter is **"n_estimators"** and controls the number of trees that are grown during the process. This means that this number represents the B iterations of the process. Increasing B generally reduces bias by allowing the model to better fit complex patterns, but excessively large values may increase variance and lead to overfitting. The second parameter is **"learning_rate"** and represents the contribution of each tree to the final model. A smaller learning rate shrinks the influence of individual trees, which helps reduce variance but requires a larger number of trees. The third parameter is **"max_depth"**. Like before, this controls the length of each individual tree and consequently the complexity. Shallow trees introduce higher bias but lower variance. The last parameter, **"subsample"**, controls the fraction of training observations (rows) randomly sampled to build each tree. Using a subsample fraction smaller than one introduces additional randomness (choice of rows), which reduces variance and improves generalization.

### 3.7.1) GradientBoosting

In [66]:
def find_params_gbr(X,y):
    
    # Model
    gbr = GradientBoostingRegressor(random_state=42)
    multi_gbr = MultiOutputRegressor(gbr)

    # Parameters grid
    param_grid = {
        "estimator__n_estimators": [300, 400, 500],
        "estimator__learning_rate": [ 0.05,0.07, 0.1],
        "estimator__max_depth": [ 4, 7, 10],
        "estimator__subsample" : [0.7,0.8,0.9,1]
    }

    grid = GridSearchCV(
        multi_gbr,
        param_grid,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        verbose=0
    )
    
    grid.fit(X, y)

    
    print("Best params:", grid.best_params_)
    print("Best MAE:", -grid.best_score_)
    
# -- CODE --  
# find_params_gbr(X,y)

# -- RESULT -- 
# Best params: {'estimator__learning_rate': 0.05, 'estimator__max_depth': 7, 'estimator__n_estimators': 500, 'estimator__subsample': 0.7}
# Best MAE: 1.9220344993886695


As an additional info : this single GridSearchCV took more than 7.5h to compute on this kaggle notebook. This just proves that searches have to be done in an optimal setting.

In [67]:
# Model
gbr = GradientBoostingRegressor(n_estimators = 500,
                                learning_rate = 0.05,
                                max_depth = 7,
                                subsample =  0.7,
                                random_state = 42
                               )

# Fit 
gbr = fit_and_cv(gbr, X,y, name='GradBoost')

The boosting model is also far better than the bagging one for every target. This boosting model seems to be better for short term forecasting but it increases in error more rapidly than the bagging model. In other words, it gains more in accuracy, compared to the bagging model, in the 12h forecast than the 24h or 48h one. Overfitting tendency still strong with tree models.

In [68]:
# Feature importances for GradientBoosting: all horizons

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

# Loop over the three targets
for i, ax in enumerate(axes):

    # Create a DataFrame with the features and the coefficients accordingly 
    results = pd.DataFrame({
        'feature': list(X.columns),
        'weight': [co for co in gbr.estimators_[i].feature_importances_]
    })

    # Sort to get the most extreme ones (both positive and negative)
    sorted_results = results.sort_values('weight', ascending=False)
    top_10 = sorted_results.head(10)
    important_results = pd.concat([top_10],axis =0)

    # Plot
    sns.barplot(
        data=important_results,
        x='weight',
        y='feature',
        ax=ax,
        palette='vlag'
    )

    # Details
    ax.set_title(f"Feature importance in GradientBoosting – Target {i+1}")
    ax.set_xlabel("Feature Importance")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


All targets are mostly dictated by temperatures. Interesting details are that the first target is also dependent on the hour, but when we look further, we don't look at the hour so much because there is no switching in night and day. Also only 1 season ? Seasons are basically reflected in the temperatures. The repartition of the importances is extremely similar to the bagging model with some variables absolutely dominating the first spot. 

In [69]:
# Plot results 
plot_results('GradBoost', 12)

In [70]:
plot_results('GradBoost', 24)

In [71]:
plot_results('GradBoost', 48)

As we look further ahead, predictability decreases and a good model that captures relations in short term won't be able to for a longer forecast. Boosting seems to be a better strategy for the first 2 targets. Compared to th ebagging model, there is a substantial decrease in variance at every level but the predictions of the extremes don't seem to be better as they are also shrunken towards the center. This means that the model performs better on mid-range temperatures (ie. most of the data). Boosting involves targeting the residuals and may be a better approach for accurate predictions.

### 3.7.2) XGBoost

XGBoost is a type of boosting, so it still builds trees sequentially to fit the residuals. However, the way the tree is built is different in XGBoost. The model grows decision trees by optimizing a similarity score for each node, a high similarity score means that the values inside a leaf are similar. To decide each split, it tries different decision parameters and calculates a gain, which is based on the similarity score. It chooses the decision split that will have a higher gain.  After building the tree, the models consider pruning a branch if the gain of the branch is less than a value (gamma). Then, like GradientBoosting, it adds a shrunken version of the tree to the model. 

In this model we have a higher number of parameters to tune so we have to use Randomized Search because Grid Search would take too much time. Some DridSearch can take hours and the kaggle notebook would crash out before it's finished. We are going to use RandomizedSearch for each target and see what the results are and then we can look at the results for all targets together. It is important to see the difference in results when using RandomizedSeach.

The first four parameters are the same as GradientBoosting. In addition, **"reg_lambda"** represents our regularization parameter. If it is high, similarity scores become smaller and gains decrease resulting in less splitting of nodes. trees bcome less aggressive and predictions get smoother (bigger leaves). If it is low, the variance-bias tradeoff goes in the other direction with more aggressive trees : lower bias, higher variance. **"gamma"** represents  our pruning treshold, the minimal gain value for a branch not to be pruned. High gamma means more pruned branches, shallow trees, higher bias, lower variance. **"colsample_bytree"** is similar to **"subsample"**, it controls the fraction of features (columns) randomly sampled to build each tree. If this measure goes towards 1, then each trees sees more features and they become more similar to each other resulting in higher bias and lower variance.

Once again, some cells are not run because the notebook wouldn't save them (too much time). The results are written in comments as --RESULT--. 

In [72]:
def find_params_xgb(X,y):

    # Model
    xgb = XGBRegressor(random_state=42)

    # Grid search
    param_grid = {
        "n_estimators":  [1000,1100, 1200,1300],
        "learning_rate": [0.01, 0.03, 0.05],
        "max_depth": [4, 6, 10],
        "subsample" : [0.8,0.9,1],
        "gamma": [0,1,2,3],
        "reg_lambda" : [1,2,5,7],
        "colsample_bytree":[0.8,0.9,1]
    }
    
    grid = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_grid,
        n_iter=50,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        verbose=0
    )

    # Fit
    grid.fit(X, y)
    
    print("Best params:", grid.best_params_)
    print("Best MAE:", -grid.best_score_)

# -- CODE -- 
# for t in target_cols:    
    # find_params_xgb(X,y[t])
    
# -- RESULT -- 
# Best params: {'reg_lambda': 5, 'reg_alpha': 1, 'n_estimators': 1300, 'max_depth': 6, 'learning_rate': 0.05, 'gamma': 0}
# Best MAE: 1.5716122369660415
# Best params: {'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 1200, 'max_depth': 6, 'learning_rate': 0.03, 'gamma': 1}
# Best MAE: 1.8252121477421936
# Best params: {'reg_lambda': 5, 'reg_alpha': 0.75, 'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.03, 'gamma': 1}
# Best MAE: 2.413593118533192

The main problem with RandomizedSearch is that we can never be sure that it actually returns the best model because the parameter picking is done randomly. If by intuition or luck, we come across other parameter settings giving better results, happy days. In general, an extensive GridSearch is a more secure path.

In [73]:
def find_params_xgb(X,y):

    # Model
    xgb = XGBRegressor(random_state=42)
    multi_xgb = MultiOutputRegressor(xgb)

    # Randomized search. Grid search would be 19440 fits... for 1 target. It doesn't finish before the 12 hours in a kaggle session.
    param_grid = {
        "estimator__n_estimators":  [1000,1100, 1200,1300],
        "estimator__learning_rate": [0.01, 0.03, 0.05],
        "estimator__max_depth": [4, 6, 10],
        "estimator__gamma": [0,1,2],
        "estimator__reg_lambda" : [1,2,5,7],
        "estimator__subsample" : [0.8,0.9,1],
        "estimator__colsample_bytree":[0.8,0.9,1]
    }
    
    grid = RandomizedSearchCV(
        estimator=multi_xgb,
        param_distributions=param_grid,
        n_iter=40,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        verbose=0
    )

    # Fit
    grid.fit(X, y)
    
    print("Best params:", grid.best_params_)
    print("Best MAE:", -grid.best_score_)
    
# -- CODE -- 
# find_params_xgb(X,y)

# -- RESULT -- 
# Best params: {'estimator__reg_lambda': 7, 'estimator__n_estimators': 1300, 'estimator__max_depth': 6, 'estimator__learning_rate': 0.03, 'estimator__gamma': 0}


In [74]:
# Model
xgb = XGBRegressor(n_estimators=1200,
                   learning_rate=0.03,
                   max_depth=6,                   
                   subsample=0.8,
                   gamma = 0,
                   reg_lambda = 1,
                   colsample_bytree=0.8,
                   random_state=42,
                   n_jobs=-1
                  )

# Fit
xgb = fit_and_cv(xgb, X,y, name='XGB')

Note : These parameter values weren't found during the RandomizedSearch even though predictions are more accurate...

This model, once again, takes the first place. It beats every other model in every time horizon. As we said for the GradientBoosting model, this one also increases rapidly in error if we look at the error difference between this model and the bagging model for every time horizon, they are respectively decreasing. A small regularization parameter coupled with a boosting method allows to accuratly target the residuals with unique trees. 

In [75]:
# Feature importances for GradientBoosting: all horizons

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

# Loop over the three targets
for i, ax in enumerate(axes):

    # Create a DataFrame with the features and the coefficients accordingly 
    results = pd.DataFrame({
        'feature': list(X.columns),
        'weight': [co for co in xgb.estimators_[i].feature_importances_]
    })

    # Sort to get the most extreme ones (both positive and negative)
    sorted_results = results.sort_values('weight', ascending=False)
    top_10 = sorted_results.head(10)
    important_results = pd.concat([top_10],axis =0)

    # Plot
    sns.barplot(
        data=important_results,
        x='weight',
        y='feature',
        ax=ax,
        palette='Purples'
    )

    # Details
    ax.set_title(f"Feature importance in XGBoost – Target {i+1}")
    ax.set_xlabel("Feature Importance")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


Similar pattern to other feature importances... these are all strictly hours, temperatures and seasons. Seasonality takes the first spot for 12h forecast.

In [76]:
# Plot results
plot_results('XGB', 12)

In [77]:
plot_results('XGB', 24)

In [78]:
plot_results('XGB', 48)

Predictions being regressed towards the mean are clearly highlighted in the first scatter plot for each time. For extreme cold temperatures, the model predicts too hot, but for high temperatures it predicts too cold. This behaviour is common in all models due the rarity of extreme temperatures. Also, leaf averages in trees smooth out the predictions. The graphs are very similar to other models, because they are siimilar models and the problems described before are general and not model specific. 

## 3.8) Random Forest ##

Last but not least, random forest. Random forest models are similar to bagging models but differ in the features available for the tree. At each split, only a random subset of features is considered. The goal is to reduce correlation between trees

We introduce 2 new parameters. **"min_samples_split"** controls the minimum number of samples required to split an internal node. Also, **"max_features"** which controls the number of features considered when looking for the best split. A smaller value of this parameter corresponds to more randomness in the choice of features, which means that trees will be more diverse and variance will decrease. Otherwise, similar trees are grown which lowers bias. 

In [79]:
# Find parameters for a random forest

def find_params_rf(X,y):
    
    # Model 
    rf = RandomForestRegressor(random_state=42, n_jobs=-1)
    model = MultiOutputRegressor(rf)
    
    # Paramter Grid
    param_grid = {
        'estimator__n_estimators': [400, 500, 600],
        'estimator__max_depth': [10,20, None],
        'estimator__min_samples_split': [1,2, 5],
        'estimator__max_features' : ["sqrt", "log2"]
    }
    
    grid_search = GridSearchCV(
        model,
        param_grid,       
        cv=5,
        scoring='neg_mean_absolute_error',  
        n_jobs=-1,
        verbose=0
    )
    
    
    grid_search.fit(X, y)
    
    
    print("Best params:", grid.best_params_)
    print("Best score  (MAE):", -grid.best_score_)

# -- CODE -- 
# find_params_rf(X,y)

# -- RESULT -- 
# Best parameters found: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 500,  'max_features' : "sqrt"}


In [80]:
# Model
rf = RandomForestRegressor(n_estimators = 500,     
                           max_depth = None,
                           min_samples_split = 2,
                           random_state = 42,
                           max_features = "sqrt",
                           n_jobs=-1            
                          )

# Fit 
rf = fit_and_cv(rf, X,y, name='RF')

Similar results to bagging for targets 2 and 3. A big difference in accuracy is shown in the first target. The additional randomness in choosing features means that we don't capture as much true signals between predictors and targets as we should. As the relations get weaker, the results get closer. 

In [81]:
# Feature importances for RandomForest: all horizons

# Create 3 plots into 1
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

# Loop over the three targets
for i, ax in enumerate(axes):

    # Create a DataFrame with the features and the coefficients accordingly 
    results = pd.DataFrame({
        'feature': list(X.columns),
        'weight': [co for co in rf.estimators_[i].feature_importances_]
    })

    # Sort to get the most extreme ones (both positive and negative)
    sorted_results = results.sort_values('weight', ascending=False)
    top_10 = sorted_results.head(10)
    important_results = pd.concat([top_10],axis =0)

    # Plot
    sns.barplot(
        data=important_results,
        x='weight',
        y='feature',
        ax=ax,
        palette='Purples'
    )

    # Details
    ax.set_title(f"Feature importance in RandomForest – Target {i+1}")
    ax.set_xlabel("Feature Importance")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


In [82]:
# Plot results 
plot_results('RF', 12)

In [83]:
plot_results('RF', 24)

In [84]:
plot_results('RF', 48)

The random forest has very similar results to bagging. Low bias, too much variance (standard deviation is high). Also, the same general struggles from other models are still shown here.

# 4) Results 

Now that every model has been fitted and tested. We can look at the results dataframes we created to have a more general look. 

## 4.1) Errors and predictions

In [85]:
# Let's add colors to see which model is best/worst:

def highlight_min_max(col):
    """
    applies green color to the column's minimum value and red to the maximum.  
    """
    min_val = col.min()
    max_val = col.max()
    
    return [
        "background-color: lightgreen" if v == min_val 
        else "background-color: lightcoral" if v == max_val
        else ""
        for v in col
    ]

In [86]:
result_error_df.style.background_gradient(cmap='coolwarm')  

In [87]:
result_error_df.style.apply(highlight_min_max, axis=0)

In [88]:
predictions_df

The 2 dataframes have been built correctly. Tree based models perform better in general because they capture non linear relations. Boosting seems to be the best short term strategy. The slight regularization seems to give the edge to XGBoost. Linear models behave similarly and regularization, even small, has negative effects on accuracy. 

## 4.2) Limitations and next steps

#### 4.2.1) Averaging the models 

We can use these dataframes to average the prediction from all models. The logic is that some models will overestimate predictions while others underestimate them, computing the average should get us more accurate predictions in general and reduce the error. 

In [89]:
def average_models_pred(result_error_df : pd.DataFrame,
                        predictions_df : pd.DataFrame,
                        weight = True,
                        models = 'all',
                        printing = False) -> list:
    """
    This function averages the prediction from different models.
    
    - weight = True -> gives weight based on errors, if false all models weight the same.
    - models = 'all' -> uses all models, otherwise expects a list of models with the names 
                corresponding to what is in the list 'models_done'
    - printing = True -> prints results in a better way.
    
    It returns a list of the shape [[models used], target1 error, target2 error, target3 error]
    """
    global models_done

    # If 'all', use all models
    if models =='all': 
        models=models_done

    # If models is a list, we make a list of the models complete names in "models_all3t"
    # and modify "result_error_df" and "predictions_df"
    else:
        
        # full name models in a list : Lin_reg -> Lin_reg_12, Lin_reg_24, Lin_reg_48
        models_all3t = [] 
        for m in models: 
            models_all3t.append(f"{m}_12")
            models_all3t.append(f"{m}_24")
            models_all3t.append(f"{m}_48")
            
        result_error_df = result_error_df[result_error_df["model"].isin(models)]                        
        predictions_df = predictions_df[[col for col in predictions_df.columns if col in models_all3t]] 
    
#---- Create weights using result_error_df 
    if weight:

        # p controls the power to which we raise errors so that the weight difference increases.
        p = 3 
        errors = result_error_df[['Test_error 12h','Test_error 24h','Test_error 48h']].values
        
        # weights has shape (m models, k targets): each model and target have a weight that corresponds to their error
        weights = (1 / errors) ** p
        weights = weights / weights.sum(axis=0)
      
        # Convert to numpy :  shape becomes (7429 rows, 9 models, 3 targets)
        preds = predictions_df.values.reshape(y.shape[0], len(models), 3)  
        
        n_rows = preds.shape[0]    # = rows = 7429
        n_models = preds.shape[1]  # = models = 9
        n_targets = preds.shape[2] # = targets = 3
        
        # Initialize result array : array with zeros in 3 columns and 7429 rows
        final_preds = np.zeros((n_rows, n_targets)) 

        # Loop through rows
        for i in range(n_rows):   
            # Loop through targets (12h, 24h, 48h)
            for t in range(n_targets): 

                # Initialize result to put in the new array "final_preds"
                weighted_sum = 0.0    

                # Loop through models
                for m in range(n_models):
                    # for this model i take his prediction (row and column) 
                    # and multiply it by his weight (doesn't change with the row)
                    weighted_sum += preds[i, m, t] * weights[m, t] 

                # Add result in the position (i, t)
                final_preds[i, t] = weighted_sum
        MAEs = [models]
        
        for i in range(y.shape[1]):
            mae = mean_absolute_error(y.iloc[:, i], final_preds[:, i])
            if printing: 
                print(f"MAE weighted for target {i} with {models}:", mae)
                print()
            MAEs.append(mae)
            
        return MAEs

#---- No weights
    else: 
        preds = predictions_df.values.reshape(y.shape[0], len(models), 3)  # (7429 rows, 9 models, 3 targets)

        # Simple average across models (axis=1)
        average_preds = preds.mean(axis=1)
        MAEs = [models]
        for i in range(y.shape[1]):
            mae = mean_absolute_error(y.iloc[:, i], average_preds[:, i])
            if printing: 
                print(f"MAE weighted for target {i} with {models}:", mae)
                print()            
            MAEs.append(mae)
            
        return MAEs

In [90]:
# These are the models that we tested:
models_done

In [91]:
# Example by averaging every model. 
average_models_pred( result_error_df, predictions_df, weight=True,models='all', printing=False)

Some models will only hurt the average. We can test every combination of model averages to see which one is the best. The next function will go trough the possible permutations of all models and print the best one for each time horizon.

In [92]:
def best_average_pred(result_error_df : pd.DataFrame,
                      predictions_df : pd.DataFrame,
                      models_done : list,
                      WEIGHT = True,
                      printing = False):
    """ 
    Returns nothing but prints the best models and errors for each of the three targets 
    expects result_error_df,predictions_df as DataFrames 
    models_done is the list of models used to make all combinations
    WEIGHT, printing= True -> will be true in average_models_pred()
    
    """
    # List of lists containing every permutation of models
    all_combinations = []
    all_combinations_result = []

    # get all combinations. Including from single models to all models.
    for r in range(1, len(models_done) + 1):
        combos = itertools.combinations(models_done, r)
        all_combinations.extend(combos)
    all_combinations = [list(c) for c in all_combinations]

    # Now we can get the result for  every combination with our "average_models_pred()"
    for i,mods in enumerate(all_combinations):
        all_combinations_result.append(average_models_pred( result_error_df, predictions_df, weight=WEIGHT,models=mods, printing=False))

    # double dictionary : first layer is what target (0,1,2),
    # second is to differentiate "score" from "models" used. 
    best = {
        0: {"score": 10000, "models": None},  # for 12h
        1: {"score": 10000, "models": None},  # for 24h
        2: {"score": 10000, "models": None},  # for 48h
    }
    
    # Through all combos
    for combo in all_combinations_result:

        # m = models used
        m = combo[0]    
        
        # 3 target scores 
        scores = combo[1:]                  

        # For target, replace score and models if new scores are better 
        for i in range(3): 
            if scores[i] < best[i]["score"]:
                best[i]["score"] = scores[i]
                best[i]["models"] = m
    
    # Print results
    print("Best for 12h:", best[0]["models"], best[0]["score"])
    print("Best for 24h:", best[1]["models"], best[1]["score"])
    print("Best for 48h:", best[2]["models"], best[2]["score"])
        

In [93]:
# Let's see the best average with weight

best_average_pred(result_error_df,predictions_df,models_done, WEIGHT=True)

In [94]:
# Let's see the best average without weight 

best_average_pred(result_error_df,predictions_df,models_done, WEIGHT=False)

Even when grouping models, tree based models are far better for this problem. 

 Results are a little better. Why doesn't averaging help our predictions more ? We said that all our models have a hard time predicting extreme values. Very high or low temperatures are underrepresented in the data. This allows for our model to predict well most of the time (a lot of residuals around 0) when  temperatures are around the mean but not extreme ones. Averaging predictions doesn't push said predictions closer to the extremes ... The two boosting models had th ebest results and averaging slightly helped.

To overcome this "extreme temperature" problem, we can try 2 last steps. Seasonal predictions and sample_weight predictions.

### 4.2.2) Seasonal Prediction 

As models struggle to predict high and low temperature. A possible way of manually helping the model is to separate the seasons and train a different model for each of them. This way, each model will only have to predict purely normal distributions.

The next function separates the rows into seasons, train a model and saves that model in a dictionary to later predict on the right season. This is done individually for every target.

In [95]:
def get_seasonal_prediction(
    X: pd.DataFrame,
    y: pd.DataFrame,
    target_cols: list,
    n_splits: int = 5,
    random_state: int = 42
):
    """
    Performs Cross-Validated seasonal prediction.

    For each fold:
    - Trains one model per season
    - Predicts season-wise on the test fold
    - Stores out-of-fold predictions

    Returns:
    - oof_predictions: DataFrame with same index as X and columns = target_cols
    - cv_mae: dictionary with MAE per target
    """

    # Define seasons (one-hot encoded)
    seasons = ["season_Winter", "season_Spring", "season_Summer", "season_Autumn"]

    # Create CV splitter
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # Store full out-of-fold predictions in a DataFrame of the same size as y.
    oof_predictions = pd.DataFrame(
        np.zeros((len(X), len(target_cols))),
        index=X.index,
        columns=target_cols
    )

    # Loop through targets
    for target in target_cols:

        # Array to store OOF predictions for this target
        oof_target = np.zeros(len(X))

        # Cross-validation loop
        for fold, (train_idx, test_idx) in enumerate(kf.split(X)):

            # Split data into train / test folds
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # Dictionary to store one fitted model per season
            season_models = {}

            # Train one model per season
            for season in seasons:

                # Select rows belonging to this season
                seasonal_rows = X_train[season] == 1

                # Seasonal training data
                X_season = X_train.loc[seasonal_rows]
                y_season = y_train.loc[seasonal_rows, target]

                # Define the model (best parameters found previously)
                model = XGBRegressor(
                    n_estimators=1100,
                    learning_rate=0.03,
                    max_depth=6,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=random_state,
                    n_jobs=-1
                )

                # Train model on this season and target
                model.fit(X_season, y_season)

                # Store fitted model
                season_models[season] = model

            # Create array to store predictions for this fold
            y_pred_fold = np.zeros(len(X_test))

            # Predict season by season on the test fold
            for season in seasons:
                seasonal_rows = X_test[season] == 1
                y_pred_fold[seasonal_rows] = season_models[season].predict(
                    X_test.loc[seasonal_rows]
                )

            # Store predictions in correct original row positions
            oof_target[test_idx] = y_pred_fold

        # Save OOF predictions for this target
        oof_predictions[target] = oof_target

    # Compute CV MAE per target
    cv_mae = {}
    for target in target_cols:
        mae = mean_absolute_error(y[target], oof_predictions[target])
        cv_mae[target] = mae
        print(f"CV MAE for {target} = {mae:.3f}")


get_seasonal_prediction(
    X,
    y,
    target_cols,
    5,
    random_state = 42
)

Results are good enough for the first 2 targets and marginally better for the third. With more data, this could be a very good strategy. 

### 4.2.3) "sample_weight" prediction

Another solution that might be better in our case is to use the parameter **"sample_weight"**: This puts a different weight for errors on some of the rows. By separating temperature targets and putting a higher weight on extreme temperatures, the model will be forced to take them more into consideration. We can decide extreme temperatures by choosing  a quantile to take from each tail of the distribution. Then next function, "fit_and_weight()", calculates MAE for the three targets given a low quantile (low_q) and a penalty (penalty). 

In [96]:
def fit_and_weight(model,
                      X : pd.DataFrame,
                      y : pd.DataFrame,
                      low_q : float, 
                      penalty : int,
                      n_splits : int = 5) -> list:
    """
    Fit one model per target with per-target sample weights
    using Cross Validation and return average MAE per target.
    """

    # We set ymmetric quantilesbecause temperature is a normal distribution
    high_q = 1 - low_q

    # Initialize CV folds
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Store MAE per fold AND per target -> we need a shape (n_splits, n_targets)
    mae_folds = np.zeros((n_splits, y.shape[1]))

    # Loop through CV folds -> 5 iterations
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):

        # Create training and testing
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Create weights matrix with same shape as y_train
        weights = np.ones((y_train.shape[0], y_train.shape[1]))

        # Compute the weights per target -> 0,target_temp_plus12 - 1,target_temp_plus24 - 2,target_temp_plus48
        for i, col in enumerate(y_train.columns):

            # Get the value from the quantile
            low = y_train[col].quantile(low_q)
            high = y_train[col].quantile(high_q)

            # Get rows where values are outside of quantiles
            extreme_rows = (y_train[col] <= low) | (y_train[col] >= high)
            weights[extreme_rows.values, i] = penalty

        # Set weights_df into the weights we just calculated
        weights_df = pd.DataFrame(weights, columns=y_train.columns)

        # Fit one model per target 
        for i, col in enumerate(y_train.columns):

            # Use as sample_weight the right column in our weights_df 
            sample_weight = weights_df.iloc[:, i].values

            # Fit
            model.fit(
                X_train,
                y_train.iloc[:, i],
                sample_weight=sample_weight
            )

            # Predict on test fold 
            y_pred = model.predict(X_test)

            # Compute MAE for this fold and this target
            mae_folds[fold, i] = mean_absolute_error(
                y_test.iloc[:, i], y_pred
            )

    # After for loop, average MAE across folds
    mean_mae = mae_folds.mean(axis=0)

    # Create a results list with our low quantile and prediction as first object and add the mean mae.
    results = [[low_q, penalty]]
    results.extend(mean_mae.tolist())

    return results


Now, as we did for the models permutations, we can look for different sets of quantiles and penalties to see which combo returns the best errors. 

In [97]:
def best_q_p(model,
                X : pd.DataFrame,
                y : pd.DataFrame,
                low_q_list : list,
                penalty_list : list,
                n_splits : int = 5):
    """
    Loop through quantile and penalty values
    and return best combinations using CV MAE.
    """

    # Initialize list that will contain every result. 
    all_combinations_result = []

    # Loop through all combinations
    for low_q in low_q_list:
        for penalty in penalty_list:

            # Add the results 
            all_combinations_result.append(
                fit_and_weight(
                    model,
                    X,
                    y,
                    low_q,
                    penalty,
                    n_splits
                )
            )

    # Store best results per target
    best = {
        0: {"score": np.inf, "models": None},  # 12h
        1: {"score": np.inf, "models": None},  # 24h
        2: {"score": np.inf, "models": None},  # 48h
    }

    # Loop through all combinations
    for combo in all_combinations_result:

        # Quantile and penalty
        m = combo[0]

        # Target scores
        scores = combo[1:]

        # Update best per target
        for i in range(3):
            if scores[i] < best[i]["score"]:
                best[i]["score"] = scores[i]
                best[i]["models"] = m

    # Print results
    print("Best for 12h:", best[0]["models"], best[0]["score"])
    print("Best for 24h:", best[1]["models"], best[1]["score"])
    print("Best for 48h:", best[2]["models"], best[2]["score"])

    return best


Try on our two best models.

In [98]:
# Model : 1
xgb = XGBRegressor(n_estimators=1200,
                   learning_rate=0.03,
                   max_depth=6,
                   subsample=0.8,
                   colsample_bytree=0.8,
                   random_state=42,
                   n_jobs=-1
                  )

# Model : 2
gbr = GradientBoostingRegressor(n_estimators = 500,
                                learning_rate = 0.05,
                                max_depth = 7,
                                subsample =  0.7,
                                random_state = 42
                               )

# Lists to iterate through
low_q_list = [0.01,0.02,0.05,0.1, 0.15]
penalty_list = [1,3,5,7,10]

In [99]:
# Get results for xgb 
best_results = best_q_p(
    model=xgb,
    X=X,
    y=y,
    low_q_list=low_q_list,
    penalty_list=penalty_list
)

In [100]:
# Get results for gbr
best_results = best_q_p(
    model=gbr,
    X=X,
    y=y,
    low_q_list=low_q_list,
    penalty_list=penalty_list
)

Slightly helps the prediction. Good results but not great, why ? First of all, extreme temperature values are not only rare, they are also intrinsically harder to predict from the available features. Another reason for that might be that we now allow for larger errors on the rest of the data. It's a tradeoff between central and tail accuracy. As we decide to penalize the tails more, we are also risking to be less accurate on the rest of the data, which is the majority. 

A last possibility to increase accuracy on extreme temperatures prediction would be to have more data, or just duplicate extreme values. However, this method would be more prone to overfitting the training data and lose accuracy on the majority of temperatures. 

# 5) Final Submission 

 The last function takes models and weights and outputs a csv final for the competition submission.

In [101]:
# to extract a csv for a submission
def fit_and_csv(models ,X_train, y_train,df_final_test,sample_weight = False, error_weight=False):
    """
    example : 
    fit_and_csv([mod1,mod2 ] ,X, y, df_final_test,sample_weight = [low_q,penalty],weight=[w1,w2])
    then download csv file called "submission" from output. 

    """
    
    # create weights as False, it changes if we gave sample_weight a list. 
    weights = False
    
    # If we give sample_weight a list:
    if isinstance(sample_weight, list):
        weights = np.ones(y_train.shape[0])

        # Take the the list and get quantile and penalty
        low_q = sample_weight[0]
        penalty = sample_weight[1]
        high_q = 1-low_q

        # Then change the weight to a penalty if the value is outside the brackets (low, high) 
        low = y_train["target_temp_plus24"].quantile(low_q)
        high = y_train["target_temp_plus24"].quantile(high_q)
        extreme_mask = (y_train["target_temp_plus24"]<=low) | (y_train["target_temp_plus24"] >= high)
        weights[extreme_mask.values] = penalty
    
    
    all_preds = []

    # For every model we gave in models: fit, predict and add the predictions to all_preds
    for model in models:
        model.fit(X_train, y_train['target_temp_plus24'], sample_weight=weights)
        y_pred = model.predict(df_final_test)
        all_preds.append(y_pred)


    
    # If we do NOT give weights to model. 
    if not error_weight: 

        
        averaged_preds = [sum(values) / len(values) for values in zip(*all_preds)]

    # If we give weights to model prediction.
    else :
        
        preds = np.array(all_preds)                      # shape = (n_models, n_samples)
        pred_weights = np.array(error_weight)            # shape = (n_models,)

         # Normalize the weights 
        pred_weights = pred_weights / pred_weights.sum()
    
        # Calculate a weighted sum across models
        averaged_preds = np.average(preds, axis=0, weights=pred_weights)

    # Put back "Id" column
    df_sub = pd.DataFrame({
        "Id" : df_test['Id'],
        "prediction": averaged_preds
    })

    # Get a csv file to submit.
    return df_sub.to_csv("submission.csv", index=False) 

In [102]:
# This is our best prediction yet : 1.7525
# model(s)
xgb = XGBRegressor(n_estimators=1100,
                   learning_rate=0.03,
                   max_depth=6,
                   subsample=0.8,
                   colsample_bytree=0.8,
                   random_state=42,
                   n_jobs=-1
                  )

gbr = GradientBoostingRegressor(n_estimators = 500,
                                learning_rate = 0.05,
                                max_depth = 7,
                                subsample =  0.7,
                                random_state = 42
                               )
# CSV: 
fit_and_csv([xgb , gbr] ,X, y,df_final_test,sample_weight = [0.01, 6], error_weight=[2,1])

Combine XGB and GradientBoosting and putting a sample weight on a very small part of predictions. 